In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import ExtraTreesClassifier

# --- 1. SETUP & LOADING ---
input_file    = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_kmeans_smote(50-20)/merged.csv'
output_folder = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_kmeans_smote(50-20)'
output_csv    = os.path.join(output_folder, 'preprocessed_dataset_with_shadow.csv')

os.makedirs(output_folder, exist_ok=True)
print(f"🚀 Loading master dataset: {input_file}")

df = pd.read_csv(input_file, engine='c', low_memory=True)
float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = df[float_cols].astype('float32')

print(f"📊 Initial shape: {df.shape}")
print(f"📋 Columns: {df.shape[1]} | Rows: {df.shape[0]:,}")

label_col = [c for c in df.columns if c.lower() == 'label'][0]

# --- 2. SUPERCLASS MAPPING ---
label_mapping = {
    'DDOS-ICMP_FLOOD': 'DDoS', 'DDOS-UDP_FLOOD': 'DDoS', 'DDOS-TCP_FLOOD': 'DDoS',
    'DDOS-PSHACK_FLOOD': 'DDoS', 'DDOS-SYN_FLOOD': 'DDoS', 'DDOS-RSTFINFLOOD': 'DDoS',
    'DDOS-SYNONYMOUSIP_FLOOD': 'DDoS', 'DDOS-UDP_FRAGMENTATION': 'DDoS', 'DDOS-ACK_FRAGMENTATION': 'DDoS',
    'DDOS-ICMP_FRAGMENTATION': 'DDoS', 'DDOS-HTTP_FLOOD': 'DDoS',
    'DOS-UDP_FLOOD': 'DoS', 'DOS-TCP_FLOOD': 'DoS', 'DOS-SYN_FLOOD': 'DoS', 'DOS-HTTP_FLOOD': 'DoS',
    'MIRAI-GREETH_FLOOD': 'Mirai', 'MIRAI-UDPPLAIN': 'Mirai', 'MIRAI-GREIP_FLOOD': 'Mirai',
    'RECON-HOSTDISCOVERY': 'Recon', 'RECON-OSSCAN': 'Recon', 'RECON-PORTSCAN': 'Recon', 'RECON-PINGSWEEP': 'Recon',
    'VULNERABILITYSCAN': 'Recon', 'SQLINJECTION': 'Web-based', 'BACKDOOR_MALWARE': 'Web-based',
    'XSS': 'Web-based', 'BROWSERHIJACKING': 'Web-based', 'COMMANDINJECTION': 'Web-based',
    'UPLOADING_ATTACK': 'Web-based', 'DNS_SPOOFING': 'Spoofing', 'MITM-ARPSPOOFING': 'Spoofing',
    'DICTIONARYBRUTEFORCE': 'Brute Force', 'BENIGN': 'Benign'
}

print("\n🧹 Mapping superclasses and creating binary labels...")
df['Superclass'] = df[label_col].astype(str).str.strip().str.upper().map(label_mapping)
df['Binary_Label'] = df['Superclass'].apply(lambda x: 0 if x == 'Benign' else 1)

print(f"\n📊 Class Distribution (Fine-grained):")
print(df[label_col].value_counts().to_string())
print(f"\n📊 Superclass Distribution:")
print(df['Superclass'].value_counts().to_string())
print(f"\n📊 Binary Label Distribution:")
print(df['Binary_Label'].value_counts().rename({0: 'Benign (0)', 1: 'Attack (1)'}).to_string())


🚀 Loading master dataset: /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_kmeans_smote(50-20)/merged.csv
📊 Initial shape: (2376628, 40)
📋 Columns: 40 | Rows: 2,376,628

🧹 Mapping superclasses and creating binary labels...

📊 Class Distribution (Fine-grained):
label
BENIGN                     1026628
DDOS-UDP_FRAGMENTATION       50000
DDOS-ACK_FRAGMENTATION       50000
RECON-PORTSCAN               50000
DNS_SPOOFING                 50000
DOS-HTTP_FLOOD               50000
DDOS-RSTFINFLOOD             50000
DDOS-SYNONYMOUSIP_FLOOD      50000
DOS-UDP_FLOOD                50000
DDOS-UDP_FLOOD               50000
DOS-SYN_FLOOD                50000
DDOS-PSHACK_FLOOD            50000
DDOS-ICMP_FLOOD              50000
DDOS-SYN_FLOOD               50000
DDOS-ICMP_FRAGMENTATION      50000
RECON-OSSCAN                 50000
DOS-TCP_FLOOD                50000
MIRAI-GREETH_FLOOD           50000
DDOS-TCP_FLOOD               50000
VULNERABILITYSCAN            50000
MITM-ARPSPOOFING         

In [6]:
# --- 3. CLEANING ---
print("\n🧹 Handling Infinity and NaN values...")
before_nan = df.isnull().sum().sum()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
numeric_cols = df.select_dtypes(include=[np.number]).columns
imputed = 0
for col in numeric_cols:
    if col not in ['Binary_Label'] and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())
        imputed += 1
after_nan = df.isnull().sum().sum()
print(f"  ✅ Columns imputed with median: {imputed}")
print(f"  ✅ NaN values before: {before_nan:,} → after: {after_nan:,}")

# --- 4. VARIANCE THRESHOLD ---
print("\n🔍 Applying Variance Threshold (removing features with < 1% variance)...")
X_raw = df.drop(columns=[label_col, 'Superclass', 'Binary_Label'])
y_bin = df['Binary_Label']

selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X_raw)
active_features = X_raw.columns[selector.get_support()].tolist()
dropped_var = len(X_raw.columns) - len(active_features)
print(f"  ✂️ Features before: {len(X_raw.columns)} | Dropped: {dropped_var} | Remaining: {len(active_features)}")

# --- 5. CORRELATION FILTER ---
print("\n🔗 Removing highly correlated features (> 0.85)...")
corr_matrix = pd.DataFrame(X_var, columns=active_features).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]
X_filtered = pd.DataFrame(X_var, columns=active_features).drop(columns=to_drop)
print(f"  ✂️ Features before: {len(active_features)} | Dropped: {len(to_drop)} | Remaining: {len(X_filtered.columns)}")

# --- 6. TOP 25 FEATURES ---
print("\n🏆 Ranking features using Extra Trees (this may take a few minutes)...")
model_fs = ExtraTreesClassifier(n_estimators=30, random_state=42, n_jobs=-1)
model_fs.fit(X_filtered, y_bin)
feat_importances = pd.Series(model_fs.feature_importances_, index=X_filtered.columns)
top_25 = feat_importances.nlargest(15).index.tolist()

print(f"\n📌 Top 15 Most Predictive Features:")
for i, (feat, imp) in enumerate(feat_importances.nlargest(15).items(), 1):
    print(f"  {i:>2}. {feat:<40} importance: {imp:.6f}")

# --- 7. ROBUST SCALING & EXPORT ---
print("\n⚖️ Applying Robust Scaling...")
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_filtered[top_25])

processed_df = pd.DataFrame(X_scaled, columns=top_25)
processed_df['Fine_Label']   = df[label_col].values
processed_df['Binary_Label'] = y_bin.values
processed_df['Superclass']   = df['Superclass'].values

processed_df.to_csv(output_csv, index=False)

print(f"\n🎉 SUCCESS!")
print(f"  📐 Final Dataset Shape: {processed_df.shape}")
print(f"  💾 Saved to: {output_csv}")
print(f"\n{'='*50}")
print(f"  Initial features : {len(X_raw.columns)}")
print(f"  After variance   : {len(active_features)}")
print(f"  After correlation: {len(X_filtered.columns)}")
print(f"  Final (Top 15)   : 25")
print(f"{'='*50}")


🧹 Handling Infinity and NaN values...
  ✅ Columns imputed with median: 3
  ✅ NaN values before: 20,180 → after: 20,000

🔍 Applying Variance Threshold (removing features with < 1% variance)...
  ✂️ Features before: 39 | Dropped: 12 | Remaining: 27

🔗 Removing highly correlated features (> 0.85)...
  ✂️ Features before: 27 | Dropped: 6 | Remaining: 21

🏆 Ranking features using Extra Trees (this may take a few minutes)...

📌 Top 15 Most Predictive Features:
   1. number                                   importance: 0.179073
   2. https                                    importance: 0.151954
   3. ack_flag_number                          importance: 0.111242
   4. time_to_live                             importance: 0.065857
   5. ack_count                                importance: 0.048499
   6. header_length                            importance: 0.045549
   7. psh_flag_number                          importance: 0.044932
   8. iat                                      importance: 0.041

In [7]:
import pandas as pd

file_path = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_kmeans_smote(50-20)/preprocessed_dataset_with_shadow.csv'
df = pd.read_csv(file_path, low_memory=False)

if 'Fine_Label' in df.columns:
    df['Fine_Label'] = df['Fine_Label'].astype('category')

duplicate_count = df.duplicated().sum()

print(f"📊 Total Rows        : {len(df):,}")
print(f"✨ Duplicate Rows    : {duplicate_count:,}")
print(f"📝 Unique Rows       : {len(df) - duplicate_count:,}")

📊 Total Rows        : 2,376,628
✨ Duplicate Rows    : 16,536
📝 Unique Rows       : 2,360,092


In [8]:
import time
import joblib
import os
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, recall_score, accuracy_score

def evaluate_edge_model_detailed(model, name, X_test, y_test, superclasses):
    # 1. Inference timing
    start_inf = time.time()
    y_pred = model.predict(X_test)
    end_inf = time.time()

    total_inference_time_s = end_inf - start_inf
    avg_latency_ms         = (total_inference_time_s / len(X_test)) * 1000
    throughput             = len(X_test) / total_inference_time_s

    # 2. Binary Metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fnr      = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0
    fpr      = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0
    accuracy = accuracy_score(y_test, y_pred) * 100

    # 3. Superclass Breakdown
    results_df = pd.DataFrame({'Actual': superclasses.values, 'Pred': y_pred})

    print(f"\n🔍 Detailed Breakdown for {name}:")
    breakdown = []
    for cat in results_df['Actual'].unique():
        subset  = results_df[results_df['Actual'] == cat]
        total   = len(subset)
        correct = (subset['Pred'] == 0).sum() if cat == 'Benign' else (subset['Pred'] == 1).sum()
        rate    = (correct / total) * 100 if total > 0 else 0
        breakdown.append({
            "Superclass": cat,
            "Samples": total,
            "Detection Rate": f"{rate:.2f}%",
            "Missed": total - correct
        })
    print(pd.DataFrame(breakdown).sort_values("Samples", ascending=False).to_string(index=False))

    # 4. Model Size
    model_filename = f"{name.strip().replace(' ', '_').lower()}.pkl"
    joblib.dump(model, model_filename)
    size_kb = os.path.getsize(model_filename) / 1024

    print(f"  ⏱️  Total Inference Time : {total_inference_time_s:.4f}s")
    print(f"  ⚡  Throughput           : {throughput:,.0f} samples/sec")
    print(f"  🎯  Accuracy             : {accuracy:.4f}%")

    return {
        "Model"               : name.strip(),
        "Accuracy (%)"        : round(accuracy, 4),
        "F1-Score"            : round(f1_score(y_test, y_pred), 4),
        "Recall (DR)"         : round(recall_score(y_test, y_pred), 4),
        "FNR"                 : f"{fnr:.3f}%",
        "FPR"                 : f"{fpr:.3f}%",
        "Latency (ms)"        : round(avg_latency_ms, 6),
        "Throughput (samp/s)" : round(throughput, 2),
        "Inference Time (s)"  : round(total_inference_time_s, 4),
        "Size (KB)"           : round(size_kb, 2)
    }

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

processed_df = pd.read_csv('/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_kmeans_smote(50-20)/preprocessed_dataset_with_shadow.csv')

X        = processed_df.drop(columns=['Binary_Label', 'Superclass', 'Fine_Label'])
y_binary = processed_df['Binary_Label']
y_fine   = processed_df['Fine_Label']
s        = processed_df['Superclass']

# Stratify on Fine_Label (34 classes) → guarantees every subclass is exactly 80/20
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y_binary, s, test_size=0.2,
    stratify=y_fine,
    random_state=42
)

print(f"✅ Train size : {len(X_train):,}")
print(f"✅ Test size  : {len(X_test):,}")
print(f"\nTrain Binary Distribution:\n{y_train.value_counts().to_string()}")
print(f"\nTest Binary Distribution:\n{y_test.value_counts().to_string()}")

models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Naïve Bayes"         : GaussianNB(),
    "Decision Tree"       : DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1),
    "Extra Trees"         : ExtraTreesClassifier(n_estimators=50, max_depth=10, n_jobs=-1),
    "AdaBoost"            : AdaBoostClassifier(n_estimators=50),
    "XGBoost"             : XGBClassifier(n_estimators=50, max_depth=6, verbosity=0),
    "LightGBM"            : lgb.LGBMClassifier(n_estimators=50, learning_rate=0.1, verbose=-1),
}

master_results = []
for name, model in models.items():
    print(f"\n🚀 Training: {name}...")
    model.fit(X_train, y_train)
    res = evaluate_edge_model_detailed(model, name, X_test, y_test, s_test)
    master_results.append(res)

print("\n" + "="*100)
print("MASTER PERFORMANCE COMPARISON")
print("="*100)
print(pd.DataFrame(master_results).sort_values(by="F1-Score", ascending=False).to_string(index=False))

✅ Train size : 1,901,302
✅ Test size  : 475,326

Train Binary Distribution:
Binary_Label
1    1080000
0     821302

Test Binary Distribution:
Binary_Label
1    270000
0    205326

🚀 Training: Logistic Regression...

🔍 Detailed Breakdown for Logistic Regression:
 Superclass  Samples Detection Rate  Missed
     Benign   205326         90.99%   18494
       DDoS   104000         99.98%      24
      Recon    44000         57.03%   18907
        DoS    40000         99.98%       7
      Mirai    30000         99.94%      19
  Web-based    24000         54.45%   10933
   Spoofing    20000         26.70%   14660
Brute Force     4000         47.88%    2085
        NaN        0          0.00%       0
  ⏱️  Total Inference Time : 0.0186s
  ⚡  Throughput           : 25,522,464 samples/sec
  🎯  Accuracy             : 86.2980%

🚀 Training: Naïve Bayes...

🔍 Detailed Breakdown for Naïve Bayes:
 Superclass  Samples Detection Rate  Missed
     Benign   205326         97.62%    4896
       DDoS   1040

In [10]:
import pandas as pd

# Reconstruct full label series aligned with the split
full_fine = processed_df['Fine_Label']

# Get train and test indices from the split
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    processed_df.index,
    test_size=0.2,
    stratify=processed_df['Fine_Label'],
    random_state=42
)

train_counts = processed_df.loc[train_idx, 'Fine_Label'].value_counts().sort_index()
test_counts  = processed_df.loc[test_idx,  'Fine_Label'].value_counts().sort_index()

summary = pd.DataFrame({
    'Train': train_counts,
    'Test':  test_counts,
    'Total': train_counts + test_counts,
    'Train %': (train_counts / (train_counts + test_counts) * 100).round(2),
    'Test %':  (test_counts  / (train_counts + test_counts) * 100).round(2),
}).sort_values('Total', ascending=False)

print(f"{'='*65}")
print(f"{'Class':<30} {'Train':>8} {'Test':>8} {'Train%':>8} {'Test%':>7}")
print(f"{'='*65}")
for label, row in summary.iterrows():
    print(f"{str(label):<30} {int(row['Train']):>8,} {int(row['Test']):>8,} {row['Train %']:>7.2f}% {row['Test %']:>6.2f}%")
print(f"{'='*65}")
print(f"{'TOTAL':<30} {train_counts.sum():>8,} {test_counts.sum():>8,}")

Class                             Train     Test   Train%   Test%
BENIGN                          821,302  205,326   80.00%  20.00%
DDOS-ACK_FRAGMENTATION           40,000   10,000   80.00%  20.00%
RECON-HOSTDISCOVERY              40,000   10,000   80.00%  20.00%
DDOS-PSHACK_FLOOD                40,000   10,000   80.00%  20.00%
DDOS-ICMP_FRAGMENTATION          40,000   10,000   80.00%  20.00%
DDOS-ICMP_FLOOD                  40,000   10,000   80.00%  20.00%
DDOS-RSTFINFLOOD                 40,000   10,000   80.00%  20.00%
DDOS-TCP_FLOOD                   40,000   10,000   80.00%  20.00%
DDOS-SYN_FLOOD                   40,000   10,000   80.00%  20.00%
DDOS-SYNONYMOUSIP_FLOOD          40,000   10,000   80.00%  20.00%
DNS_SPOOFING                     40,000   10,000   80.00%  20.00%
DDOS-UDP_FRAGMENTATION           40,000   10,000   80.00%  20.00%
DDOS-UDP_FLOOD                   40,000   10,000   80.00%  20.00%
MIRAI-GREETH_FLOOD               40,000   10,000   80.00%  20.00%
DOS-UDP_FL